In [2]:
import pandas as pd
from pathlib import Path

In [5]:
# Checking that speed buckets are cumulative (monotonically decreasing) 
from network_idx.constants import STATE_USPS_TO_FIPS

# Check that each speed bucket is cumulative (monotonically decreasing)
speed_cols = [
    "speed_02_02",
    "speed_10_1",
    "speed_25_3",
    "speed_100_20",
    "speed_250_25",
    "speed_1000_100"
]

# For each row, check that the values are non-increasing
def is_cumulative(row):
    return all(row[speed_cols[i]] >= row[speed_cols[i+1]] for i in range(len(speed_cols)-1))
i=1
total_states = len(STATE_USPS_TO_FIPS)
for state, fips in STATE_USPS_TO_FIPS.items():
    csv_file = Path(f"/home/eprashar_solutions_corelogic_com/network-idx/data/extracted/fcc/broadband_coverage/{state}/bdc_{fips}_fixed_broadband_summary_by_geography_place_J25_31mar2026.csv")
    df = pd.read_csv(csv_file)
    df["cumulative_check"] = df.apply(is_cumulative, axis=1)
    
    # Compute MECE buckets
    df["less_than_100_20"] = df["speed_02_02"] - df["speed_100_20"]
    df["speed_100_20_only"] = df["speed_100_20"] - df["speed_250_25"]
    df["more_than_100_20"] = df["speed_250_25"]

    # Check that the buckets are MECE and exhaustive
    df["mece_sum"] = df["less_than_100_20"] + df["speed_100_20_only"] + df["more_than_100_20"]
    df["mece_check"] = (df["mece_sum"] - df["speed_02_02"]).abs() < 1e-6  # allow for float error 
    i+=1 
    # Print summary
    print(df.head(5))
    #print(f"{i}/{total_states}: for state {state}, MECE sum is {df['mece_sum']}")
    #print(f"{i}/{total_states}: for state {state}, rows where MECE buckets do not sum to speed_02_02:", (~df["mece_check"]).sum())
    #print(f"{i}/{total_states}: for state {state}, rows where speed buckets are not cumulative:", (~df["cumulative_check"]).sum())
    # Optionally, show a few rows for visual inspection
    # print(df[["less_than_100_20", "speed_100_20_only", "more_than_100_20", "mece_sum", "speed_02_02"]].head())
    
    
    #print(df.loc[~df["cumulative_check"], speed_cols + ["geography_id", "technology"]])

    # Optionally, show a few rows for visual inspection
    # print(df[speed_cols].head())


  area_data_type geography_type  geography_id geography_desc  \
0          Total   Census Place        100100         Abanda   
1          Total   Census Place        100100         Abanda   
2          Total   Census Place        100100         Abanda   
3          Total   Census Place        100100         Abanda   
4          Total   Census Place        100100         Abanda   

  geography_desc_full  total_units biz_res       technology  speed_02_02  \
0          Abanda, AL           97       R   Any Technology     1.000000   
1          Abanda, AL           97       B   Any Technology     1.000000   
2          Abanda, AL           97       R        All Wired     0.257732   
3          Abanda, AL           97       B        All Wired     0.257732   
4          Abanda, AL           97       R  Any Terrestrial     0.257732   

   speed_10_1  speed_25_3  speed_100_20  speed_250_25  speed_1000_100  \
0         1.0         1.0           1.0           1.0             0.0   
1         1.

KeyboardInterrupt: 

In [2]:
parquet_file = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/block/fcc_coverage_block_AL_01.parquet")
df = pd.read_parquet(parquet_file)
print(df.info())
print(df.head(25))

<class 'pandas.DataFrame'>
RangeIndex: 185976 entries, 0 to 185975
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   block_geoid              185976 non-null  str    
 1   state_fips               185976 non-null  str    
 2   state_usps               185976 non-null  str    
 3   county_geoid             185976 non-null  str    
 4   tract_geoid              185976 non-null  str    
 5   place_geoid              101328 non-null  str    
 6   source                   185976 non-null  str    
 7   census_housing_units     185976 non-null  int64  
 8   estimated_fcc_units      185976 non-null  int64  
 9   copper_speed_100_20      129616 non-null  float64
 10  copper_less_than_100_20  129616 non-null  float64
 11  copper_more_than_100_20  129616 non-null  float64
 12  cable_speed_100_20       129616 non-null  float64
 13  cable_less_than_100_20   129616 non-null  float64
 14  cable_more_than

In [6]:
# Tract level file
tract_parquet = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/features/fcc/broadband_coverage/tract/fcc_coverage_tract_AK_02.parquet")
tract_df = pd.read_parquet(tract_parquet)
print(tract_df.info())
print(tract_df.head(25))

<class 'pandas.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   tract_geoid                     177 non-null    str    
 1   state_fips                      177 non-null    str    
 2   state_usps                      177 non-null    str    
 3   estimated_census_housing_units  177 non-null    int64  
 4   estimated_fcc_units             177 non-null    int64  
 5   copper_speed_100_20             177 non-null    float64
 6   copper_less_than_100_20         177 non-null    float64
 7   copper_more_than_100_20         177 non-null    float64
 8   cable_speed_100_20              177 non-null    float64
 9   cable_less_than_100_20          177 non-null    float64
 10  cable_more_than_100_20          177 non-null    float64
 11  fiber_speed_100_20              177 non-null    float64
 12  fiber_less_than_100_20          177 non-null   